# Publish distilled adapters to HuggingFaceUploads the four Phase 2-beta adapters as `ArgParser-v1` through `ArgParser-v4`. Requires `huggingface-cli login` with a write-scope token first.

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
python3 <<'PY'
from huggingface_hub import whoami, create_repo, upload_folder
from pathlib import Path

me = whoami()
username = me['name']
print(f"logged in as: {username}\n")

UPLOADS = [
    ("phase2_student_beta_qwen1.5b_lora",    "argrag-phase2beta-v2",
     "v2: LoRA r=16 on 4 gold corpora (AAEC/AbstRCT/CDCP/PERSPECTRUM/Microtext), fresh"),
    ("phase2_student_beta_qwen1.5b_lora_v3", "argrag-phase2beta-v3",
     "v3: continual from v2 + AAEC, 1 additional epoch"),
    ("phase2_student_beta_qwen1.5b_lora_v4", "argrag-phase2beta-v4",
     "v4: fresh on 5 gold + LIARArg silver (2,123 rows) from gpt-oss-120b + CoT"),
]

for local_dir, repo_name, summary in UPLOADS:
    d = Path(local_dir)
    if not d.exists():
        print(f"⚠️  skipping {local_dir} — dir not found\n"); continue

    # Placeholder README (thorough model card comes later)
    placeholder = f"""---
license: apache-2.0
base_model: Qwen/Qwen2.5-1.5B-Instruct
library_name: peft
tags:
  - argument-mining
  - fact-checking
  - lora
  - qwen
language: [en]
pipeline_tag: text-generation
---

# ArgRAG {repo_name}

**Placeholder — full model card coming.**

{summary}

- Base: `Qwen/Qwen2.5-1.5B-Instruct`
- Method: LoRA r=16, alpha=32, target `q_proj,k_proj,v_proj,o_proj`
- Repository: (link to be added)

Detailed evaluation, usage, and limitations to be added.
"""
    (d / "README.md").write_text(placeholder)

    repo_id = f"{username}/{repo_name}"
    create_repo(repo_id, private=True, exist_ok=True, repo_type="model")
    print(f"→ pushing {local_dir} → {repo_id} ...", flush=True)
    upload_folder(
        repo_id=repo_id,
        folder_path=str(d),
        repo_type="model",
        commit_message=f"Initial upload: {summary[:60]}",
    )
    files = [p.name for p in d.iterdir() if p.is_file()]
    size_mb = sum(p.stat().st_size for p in d.iterdir() if p.is_file()) / 1024 / 1024
    print(f"  ✓ {len(files)} files, {size_mb:.1f} MB")
    print(f"  https://huggingface.co/{repo_id}\n")

print("=== done — all 3 adapters uploaded private ===")
print("Flip to public via web UI when ready: Settings → Change visibility")
PY

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
python3 <<'PY'
from huggingface_hub import whoami, create_repo, upload_folder
from pathlib import Path
username = whoami()['name']

d = Path("phase2_student_beta_qwen0.5b")
if d.exists():
    placeholder = """---
license: apache-2.0
base_model: Qwen/Qwen2.5-0.5B-Instruct
library_name: transformers
tags: [argument-mining, fact-checking, distillation, qwen]
language: [en]
pipeline_tag: text-generation
---

# ArgRAG Phase 2-β v1 — Qwen-0.5B full fine-tune

**Placeholder — full model card coming.**

v1 baseline: full fine-tune of Qwen-0.5B on 4 gold arg-mining corpora.
Note: this is a full-model release (~1 GB), not a LoRA adapter.
"""
    (d / "README.md").write_text(placeholder)
    repo_id = f"{username}/argrag-phase2beta-v1"
    create_repo(repo_id, private=True, exist_ok=True, repo_type="model")
    print(f"→ pushing v1 (~1 GB, may take a few minutes)...")
    upload_folder(repo_id=repo_id, folder_path=str(d), repo_type="model",
                  commit_message="Initial upload: v1 Qwen-0.5B full FT baseline")
    print(f"✓ https://huggingface.co/{repo_id}")
else:
    print("v1 dir not found — skipping")
PY

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
python3 <<'PY'
from huggingface_hub import whoami, HfApi, create_repo, upload_folder
from pathlib import Path

api = HfApi()
username = whoami()['name']
print(f"logged in as: {username}\n")

MAP = [
    ("phase2_student_beta_qwen0.5b",
     "argrag-phase2beta-v1", "ArgParser-v1",
     "v1: Qwen-0.5B full fine-tune on 4 gold corpora"),
    ("phase2_student_beta_qwen1.5b_lora",
     "argrag-phase2beta-v2", "ArgParser-v2",
     "v2: Qwen-1.5B + LoRA r=16 on 4 gold, fresh, 3 epochs"),
    ("phase2_student_beta_qwen1.5b_lora_v3",
     "argrag-phase2beta-v3", "ArgParser-v3",
     "v3: continual from v2 + AAEC, 1 additional epoch"),
    ("phase2_student_beta_qwen1.5b_lora_v4",
     "argrag-phase2beta-v4", "ArgParser-v4",
     "v4: fresh on 5 gold + LIARArg silver (2,123 rows) from gpt-oss-120b + CoT"),
]

def exists(repo_id):
    try:
        api.repo_info(repo_id, repo_type="model")
        return True
    except Exception:
        return False

for local_dir, old_name, new_name, summary in MAP:
    old_id = f"{username}/{old_name}"
    new_id = f"{username}/{new_name}"

    if exists(new_id):
        print(f"✓ {new_id} already exists — skipping\n")
        continue

    if exists(old_id):
        print(f"→ renaming {old_id} → {new_id}")
        try:
            api.move_repo(from_id=old_id, to_id=new_id, repo_type="model")
            print(f"  ✓ renamed. https://huggingface.co/{new_id}\n")
            continue
        except Exception as e:
            print(f"  ⚠️  rename failed ({e}); falling through to upload with new name\n")

    d = Path(local_dir)
    if not d.exists():
        print(f"⚠️  {local_dir} not found locally — skipping\n"); continue

    is_full_ft = "0.5b" in local_dir.lower()
    lib = "transformers" if is_full_ft else "peft"
    base = "Qwen/Qwen2.5-0.5B-Instruct" if is_full_ft else "Qwen/Qwen2.5-1.5B-Instruct"

    lora_tag = "" if is_full_ft else "\n  - lora"
    placeholder = f"""---
license: apache-2.0
base_model: {base}
library_name: {lib}
tags:
  - argument-mining
  - fact-checking
  - qwen{lora_tag}
language: [en]
pipeline_tag: text-generation
---

# ArgParser — {new_name}

**Placeholder — thorough model card coming.**

{summary}

Detailed evaluation, usage, and limitations to be added.
"""
    (d / "README.md").write_text(placeholder)

    print(f"→ uploading {local_dir} → {new_id}")
    create_repo(new_id, private=True, exist_ok=True, repo_type="model")
    upload_folder(
        repo_id=new_id,
        folder_path=str(d),
        repo_type="model",
        commit_message=f"Initial upload: {summary[:60]}",
    )
    size_mb = sum(p.stat().st_size for p in d.iterdir() if p.is_file()) / 1024 / 1024
    print(f"  ✓ uploaded ({size_mb:.1f} MB). https://huggingface.co/{new_id}\n")

print("=== done ===")
print("All 4 models under new naming:")
for _, _, new_name, _ in MAP:
    print(f"  https://huggingface.co/{username}/{new_name}")
PY

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
python3 <<'PY'
from huggingface_hub import whoami, upload_file
from pathlib import Path

username = whoami()['name']
print(f"pushing READMEs for user: {username}\n")

# ─────────────────────────── v4 — the long one ───────────────────────────
V4_README = f'''---
license: apache-2.0
base_model: Qwen/Qwen2.5-1.5B-Instruct
library_name: peft
tags:
  - argument-mining
  - fact-checking
  - lora
  - qwen
  - distillation
language: [en]
pipeline_tag: text-generation
---

# ArgParser-v4

A Qwen-1.5B LoRA that extracts argument structure — claims, premises,
citations, and support/attack relations — from political claims and
argumentative prose. This is the fourth iteration of a distillation
project, and the one that actually works well enough to plug into a
real fact-checking pipeline.

## Where this came from

I built an argument-aware retrieval pipeline for Politifact-style
fact-checking. Given a claim like "Politician X said Y," it retrieves
evidence using argument-role-targeted queries and returns a 6-way truth
verdict. When the parser was reading gold argument annotations from the
LIARArg dataset, this pipeline hit 0.422 6-way F1 versus 0.114 for a
flat-RAG baseline. The 0.308 gap was the whole reason to build it.

But that gap only means something if a real parser can slot in at
inference time. Gold annotations aren't available for actual incoming
claims. So the question became: what parser closes that gap?

First cut, Phase 2-α, was `gpt-oss-120b` running zero-shot on Cerebras.
It closed about 45% of the gap (integration F1 0.254). Real, useful,
but calling a 120B cloud model per claim isn't practical for anything
resembling deployment.

Phase 2-β was the distillation project. Four iterations. Can a small
local model preserve most of the gain?

## The four iterations

**v1** was Qwen-0.5B, full fine-tune on four argument-mining corpora
(AbstRCT, Microtext, CDCP, PERSPECTRUM), 1,494 records, 3 epochs. The
smallest reasonable baseline. In-domain comp-F1 averaged 0.108 across
the four held-out test sets. High empty rates on some domains
(PERSPECTRUM: 91% empty). The point was to get the pipeline plumbing
right, not to publish a number.

**v2** kept the same data but moved to Qwen-1.5B with LoRA r=16
(α=32, dropout 0.05, target `q_proj,k_proj,v_proj,o_proj`). 3 epochs.
In-domain comp-F1: 0.219, roughly double v1. Microtext premise F1
jumped from 0.000 to 0.680. AbstRCT empty rate 75% → 50%. Scale +
LoRA + longer training context are the dominant levers here.

**v3** was v2's adapter continued for one more epoch after adding a
fifth corpus (AAEC, 402 persuasive essays). Marginal in-domain
improvement (0.229). PERSPECTRUM actually regressed slightly, which
was the first sign that adding more of the same kind of extractive
gold hits diminishing returns quickly. Then I tried v3 on the actual
LIARArg parse and hit 83% empty rate on the first 64 rows. Killed
that run. The lesson was clear: extractive gold from academic
argument-mining corpora doesn't teach a small student to handle
Politifact-style claims. The distribution gap is too wide.

**v4** is this model. Two changes from v3:

1. Fresh adapter, not continual. Clean A/B against v3.
2. I generated 2,123 silver labels on LIARArg train articles using
   `gpt-oss-120b` via Cerebras. Both the extracted argument structure
   and the model's Chain-of-Thought reasoning came back (the CoT was
   in `message.reasoning`, which I captured almost by accident). The
   training code's target-formatter passes reasoning through as
   `<think>...</think>{{json}}` when present, so v4 accidentally
   became CoT-aware for LIARArg-style inputs — while staying purely
   extractive on the gold in-domain corpora (where reasoning was empty).

This wasn't planned. It just happened, and it turned out to be exactly
what was missing.

Training details: 3,617 records total, 3 epochs, fresh LoRA adapter,
fp16, Adafactor, gradient checkpointing. Batch 1, grad accum 32.
About 29 hours on a single GTX 1080 Ti.

## What v4 actually does

The load-bearing number is Phase 1 integration on LIARArg — the whole
reason to build any of this:

| Metric | v4 | Phase 2-α teacher (120B) | Teacher retention |
|---|---|---|---|
| 6-way F1 | 0.217 | 0.254 | 85% |
| 3-way F1 | 0.457 | 0.461 | 99% |
| within-1 accuracy | 0.605 | 0.616 | 98% |

Flat-RAG baseline for reference: 0.114. v4 beats it by 0.103 and
closes 33% of the gold-parser gap using a locally-runnable 1.5B model.

LIARArg parse empty rate went from v3's 83% down to 23%. Silver + CoT
was the missing piece.

In-domain retention is modest — 0.192 comp-F1 averaged across the five
training corpora, slightly regressed from v3's 0.229. That's the cost
of a fresh 3-epoch run versus v3's effective 4 epochs. The trade was
worth it for the cross-domain transfer.

## OOD probes

To characterize where v4 stops transferring, I ran three unseen-domain
probes:

**AMPERSAND** (Chakrabarty et al. 2019, Reddit ChangeMyView). Binary
is-argumentative F1 = 0.819 with recall 0.970 on 150 balanced
sentences. v4 catches essentially all argumentative content in Reddit
debate. The 30% false-positive rate comes from over-flagging fragments
and borderline sentences; some of those are arguably right and just
disagree with the annotator.

**PERSUADE 2.0** (Kaggle Feedback Prize, student argumentative essays).
Component F1 macro = 0.193 with 45.7% extraction rate on 25 essays.
Claim F1 = 0.351, premise F1 = 0.034. v4 finds about half of
PERSUADE's argumentative spans and labels claim-like content
reasonably. Premise F1 collapses because PERSUADE's Evidence and
Rebuttal categories are much narrower than v4's premise concept.

**ECHR** (European Court of Human Rights case briefs). Component F1
= 0.074 with 9.7% extraction rate against a proxy gold derived from
ECHR's agent labels. Legal reasoning is structurally distant from
anything v4 saw in training.

Ordering (Reddit > essays > legal) tracks discourse-register proximity
to v4's training data. Predictable, but useful to have quantified.

## Usage

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, re, json

base_id    = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_id = "{username}/ArgParser-v4"

tok = AutoTokenizer.from_pretrained(base_id)
base = AutoModelForCausalLM.from_pretrained(
    base_id, torch_dtype=torch.float16, device_map="auto"
)
model = PeftModel.from_pretrained(base, adapter_id)

INSTR = ("Extract all argument components and relations from the text. "
         "Output strict JSON with claim_components, premise_components, "
         "citation_components, and relations.")

def parse(text, max_new_tokens=1024):
    prompt = tok.apply_chat_template(
        [{{"role": "user", "content": f"{{INSTR}}\\n\\nTEXT:\\n{{text}}"}}],
        tokenize=False, add_generation_prompt=True,
    )
    enc = tok(prompt, return_tensors="pt", truncation=True,
              max_length=2048).to(model.device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens,
                         do_sample=False,
                         pad_token_id=tok.pad_token_id,
                         eos_token_id=tok.eos_token_id)
    raw = tok.decode(out[0, enc["input_ids"].shape[-1]:],
                     skip_special_tokens=True)
    cleaned = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL)
    m = re.search(r"\\{{.*\\}}", cleaned, flags=re.DOTALL)
    return (json.loads(m.group()) if m else None), raw

pred, raw = parse(
    "The Obama administration is putting Border Patrol agents in a chokehold."
)
print(pred)
```

## Things worth knowing before using this

The model emits `<think>...</think>` blocks on political and
opinionated inputs (that's where it saw CoT during training). On
formal or structured text — legal writing, some scientific abstracts
— it goes straight to JSON. Neither is a bug, it just needs handling
if you're stripping the raw generation.

Long inputs can trigger a relation-generation loop that leaves the JSON
unclosed. What happens: the model emits valid `claim_components` and
`premise_components` early, then gets stuck emitting
`{{"src": N, "tgt": M, "type": "support"}}` tuples in a repeating
pattern until it hits `max_new_tokens`. You never see the closing `}}`
so strict JSON parsing rejects everything. The workaround is either
setting `max_new_tokens` conservatively, or using a lenient parser
that pulls each valid `{{...}}` object out of each section
independently and dedupes relations. I use the second approach in
the training repo.

v4 systematically over-predicts spans on OOD text — precision runs
lower than recall in every probe. On borderline sentences it defaults
to labeling as claim. Something to keep in mind if downstream
consumers care about precision.

Fine-grained annotation schemas map imperfectly to v4's binary
claim/premise split. PERSUADE's Evidence/Rebuttal distinction and
ECHR's Court/Applicant/State agent labels don't translate directly.
v4 knows a claim from a premise; it doesn't know PERSUADE's Evidence
from PERSUADE's Rebuttal.

## Training summary

- Base: `Qwen/Qwen2.5-1.5B-Instruct`
- LoRA: r=16, α=32, dropout 0.05, on `q_proj,k_proj,v_proj,o_proj`
- Training records: 3,617 (5 gold corpora + 2,123 LIARArg silver)
- Epochs: 3, fresh adapter
- Optimizer: Adafactor, fp16, gradient checkpointing
- Hardware: single NVIDIA GTX 1080 Ti
- Wall clock: ~29 h

## License

Apache 2.0. Base model (Qwen 2.5) is also Apache 2.0. Use however you
want, no warranty.
'''

# ────────────────────────── shorter READMEs ──────────────────────────
V1_README = f'''---
license: apache-2.0
base_model: Qwen/Qwen2.5-0.5B-Instruct
library_name: transformers
tags: [argument-mining, fact-checking, qwen]
language: [en]
pipeline_tag: text-generation
---

# ArgParser-v1

Baseline for the ArgParser series. Full fine-tune of Qwen-0.5B on four
argument-mining corpora (AbstRCT, Microtext, CDCP, PERSPECTRUM), 1,494
records total, 3 epochs, fp16, Adafactor. About 1.5 hours on a
GTX 1080 Ti.

Held-out component-F1 averaged across the four domains: **0.108**.
Best on CDCP claim extraction (0.501). Worst on PERSPECTRUM (91%
empty rate — the debate-text format defeats extractive parsing here).

This is the smallest useful reference point. Kept up mostly for
reproducibility of the ablation series. If you want to actually use
one of these, use [ArgParser-v4]({{}}) — same repo family, gets a
real Phase 1 integration F1 of 0.217 versus this baseline's near-zero
usefulness on that task.

## Config

- Base: `Qwen/Qwen2.5-0.5B-Instruct`
- Method: full fine-tune, 494M trainable params
- Data: 4 gold argument-mining corpora, 1,494 records
- Epochs: 3
- Wall clock: 1.5 h on GTX 1080 Ti

## License

Apache 2.0.
'''.format(f"https://huggingface.co/{username}/ArgParser-v4")

V2_README = f'''---
license: apache-2.0
base_model: Qwen/Qwen2.5-1.5B-Instruct
library_name: peft
tags: [argument-mining, fact-checking, lora, qwen]
language: [en]
pipeline_tag: text-generation
---

# ArgParser-v2

Same training data as v1 (four argument-mining corpora, 1,494 records)
but a larger base and LoRA instead of full fine-tune. Qwen-1.5B with
LoRA r=16 (α=32, dropout 0.05, target `q_proj,k_proj,v_proj,o_proj`).
3 epochs, about 13.5 hours on a GTX 1080 Ti.

Held-out component-F1: **0.219** — roughly double v1. Microtext premise
F1 went from 0.000 to 0.680. AbstRCT empty rate 75% → 50%. Scale plus
LoRA plus longer training context are the dominant levers, and this
was the run that made that obvious.

Not the best model in the series. For actual use pick
[ArgParser-v4]({{}}) instead — it adds cross-domain transfer to
LIARArg-style political claims, which is what most people probably
care about.

## Config

- Base: `Qwen/Qwen2.5-1.5B-Instruct`
- Method: LoRA r=16 (4.4M trainable params)
- Data: 4 gold corpora, 1,494 records
- Epochs: 3
- Wall clock: 13.5 h

## Usage

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM
base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model = PeftModel.from_pretrained(base, "{username}/ArgParser-v2")
```

## License

Apache 2.0.
'''.format(f"https://huggingface.co/{username}/ArgParser-v4")

V3_README = f'''---
license: apache-2.0
base_model: Qwen/Qwen2.5-1.5B-Instruct
library_name: peft
tags: [argument-mining, fact-checking, lora, qwen]
language: [en]
pipeline_tag: text-generation
---

# ArgParser-v3

v2's adapter continued for one more epoch after adding a fifth corpus:
AAEC (402 persuasive essays, ~6000 argument components). ~5.5 hours
on the same GTX 1080 Ti.

Held-out component-F1: **0.229**, a marginal improvement over v2's
0.219. Microtext and AbstRCT nudged up; PERSPECTRUM slightly regressed
(0.056 → 0.034). Adding more of the same kind of extractive academic
gold hits diminishing returns pretty quickly.

I also tried v3 on the actual LIARArg parse — the whole point of the
project — and hit an **83% empty rate** on the first 64 rows. Real
outputs were fragmentary ("is not clear" as a claim). Killed the run
after that; it was obvious this variant couldn't do cross-domain
transfer to Politifact-style claims. The five academic argument-mining
corpora aren't enough on their own to bridge that gap.

That result motivated [v4]({{}}) — adding silver labels from a large
teacher (`gpt-oss-120b`) on 2,123 LIARArg training articles, with
Chain-of-Thought reasoning traces preserved through training. v4 gets
Phase 1 integration F1 = 0.217, closes 33% of the gold-parser gap.

For actual use, go to v4. This one exists for the ablation record.

## Config

- Base: `Qwen/Qwen2.5-1.5B-Instruct`
- Method: LoRA r=16, continual from v2
- Data: 5 gold corpora (AAEC added), 1,823 records
- Epochs: 1 continual (~4 epochs of learning total including v2)
- Wall clock: 5.5 h

## Usage

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM
base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model = PeftModel.from_pretrained(base, "{username}/ArgParser-v3")
```

## License

Apache 2.0.
'''.format(f"https://huggingface.co/{username}/ArgParser-v4")

# ─────────────────────────── write + push ───────────────────────────
READMES = [
    ("phase2_student_beta_qwen0.5b",       "ArgParser-v1", V1_README),
    ("phase2_student_beta_qwen1.5b_lora",  "ArgParser-v2", V2_README),
    ("phase2_student_beta_qwen1.5b_lora_v3","ArgParser-v3", V3_README),
    ("phase2_student_beta_qwen1.5b_lora_v4","ArgParser-v4", V4_README),
]
for local_dir, repo_name, content in READMES:
    d = Path(local_dir)
    if not d.exists():
        print(f"⚠️  {local_dir} not found — skipping\\n"); continue
    (d / "README.md").write_text(content)
    repo_id = f"{username}/{repo_name}"
    upload_file(
        path_or_fileobj=str(d / "README.md"),
        path_in_repo="README.md",
        repo_id=repo_id,
        repo_type="model",
        commit_message="Update README",
    )
    print(f"✓ {repo_id} ({len(content)} chars)")

print("\\n=== all READMEs pushed ===")
print(f"→ v4 (long): https://huggingface.co/{username}/ArgParser-v4")
print(f"→ v1/v2/v3: brief, all link back to v4")
PY

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
python3 <<'PY'
from huggingface_hub import whoami, create_collection, add_collection_item
username = whoami()['name']

col = create_collection(
    title="ArgParser: Distilled Argument-Structure Extractors",
    description="Distilling gpt-oss-120b's argument extraction into small Qwen adapters. v4 retains 85% of the teacher F1 on Politifact-style claims.",
)
print(f"collection: {col.slug}")

for name in ["ArgParser-v1", "ArgParser-v2", "ArgParser-v3", "ArgParser-v4"]:
    add_collection_item(collection_slug=col.slug,
                        item_id=f"{username}/{name}", item_type="model")
    print(f"  added {name}")

print(f"\n→ https://huggingface.co/collections/{col.slug}")
PY